# Distribución de glaciares por tipo y región en Chile (IPG 2022)
**Proyecto:** Deshielo de glaciares y crisis hídrica en Chile  
**Integrante:** Clarita Díaz  
**Fuente:** Inventario Público de Glaciares 2022 (IPG 2022), DGA-MOP

In [1]:
# Instalar Altair si es necesario
!pip install altair -q

zsh:1: command not found: pip


In [2]:
import pandas as pd
import altair as alt

# Cargar datos
df = pd.read_csv('IPG2022_limpio.csv')
print('Filas:', len(df))
df.head()

PermissionError: [Errno 1] Operation not permitted: 'IPG2022_limpio.csv'

In [ ]:
# Ver tipos de glaciar disponibles
print('Tipos de glaciar:')
print(df['tipo_glaciar'].value_counts())
print('\nRegiones:')
print(df['region'].value_counts())

In [ ]:
# Procesar: contar glaciares y sumar área por región y tipo
df['tipo_glaciar'] = df['tipo_glaciar'].str.strip().str.title()
df['region'] = df['region'].str.strip().str.title()

# Agrupar por región y tipo
agrupado = df.groupby(['region', 'tipo_glaciar']).agg(
    cantidad=('codigo_glaciar', 'count'),
    area_total_km2=('area_km2', 'sum')
).reset_index()

# Orden de regiones de norte a sur
orden_regiones = [
    'Arica Y Parinacota', 'Tarapaca', 'Antofagasta', 'Atacama',
    'Coquimbo', 'Valparaiso', 'Metropolitana De Santiago',
    'Del Maule', 'Nuble', 'Biobio', 'Araucania',
    'Los Rios', 'Los Lagos',
    'Aysen Del General Carlos Ibanez Del Campo',
    'Magallanes Y Antartica Chilena'
]

# Colores por tipo de glaciar
colores = {
    'Glaciar De Montaña': '#2166ac',
    'Glaciarete': '#74add1',
    'Glaciar De Valle': '#313695',
    'Glaciar Efluente': '#abd9e9',
    'Glaciar Rocoso': '#4d9221'
}

print('Procesamiento listo. Filas agrupadas:', len(agrupado))
agrupado.head(10)

In [ ]:
# ── VISUALIZACIÓN: Barras apiladas por región y tipo de glaciar ──

chart = alt.Chart(agrupado).mark_bar().encode(
    x=alt.X('region:N',
            sort=orden_regiones,
            title='Región',
            axis=alt.Axis(labelAngle=-45, labelLimit=200)),
    y=alt.Y('cantidad:Q',
            title='Número de glaciares',
            stack='zero'),
    color=alt.Color('tipo_glaciar:N',
                    title='Tipo de glaciar',
                    scale=alt.Scale(
                        domain=list(colores.keys()),
                        range=list(colores.values())
                    )),
    tooltip=[
        alt.Tooltip('region:N', title='Región'),
        alt.Tooltip('tipo_glaciar:N', title='Tipo'),
        alt.Tooltip('cantidad:Q', title='N° glaciares'),
        alt.Tooltip('area_total_km2:Q', title='Área total (km²)', format='.2f')
    ]
).properties(
    title=alt.TitleParams(
        text='Distribución de glaciares por tipo y región en Chile (2022)',
        subtitle='Fuente: Inventario Público de Glaciares 2022, DGA-MOP',
        fontSize=16,
        subtitleFontSize=12,
        anchor='start'
    ),
    width=750,
    height=420
).configure_axis(
    labelFontSize=11,
    titleFontSize=12
).configure_legend(
    labelFontSize=11,
    titleFontSize=12
)

chart

In [ ]:
# ── EXPORTAR HTML ──
chart.save('glaciares_por_tipo_region.html')
print('✅ HTML exportado: glaciares_por_tipo_region.html')

In [ ]:
# ── EXPORTAR JPG ──
# Instalar dependencias para exportar imagen
!pip install vl-convert-python -q

chart.save('glaciares_por_tipo_region.jpg')
print('✅ JPG exportado: glaciares_por_tipo_region.jpg')

In [ ]:
# ── PREGUNTAS QUE RESPONDE LA VISUALIZACIÓN ──

# Pregunta 1: ¿Qué región tiene más glaciares?
top_region = agrupado.groupby('region')['cantidad'].sum().sort_values(ascending=False).head(5)
print('Top 5 regiones por número de glaciares:')
print(top_region)

# Pregunta 2: ¿Qué tipo de glaciar es más común en Chile?
print('\nTipo de glaciar más frecuente:')
print(agrupado.groupby('tipo_glaciar')['cantidad'].sum().sort_values(ascending=False))

# Pregunta 3: ¿Qué región tiene más glaciares rocosos?
rocosos = agrupado[agrupado['tipo_glaciar'] == 'Glaciar Rocoso'].sort_values('cantidad', ascending=False)
print('\nGlaciares rocosos por región:')
print(rocosos[['region','cantidad']].head(5))

In [ ]:
 ── VISUALIZACIÓN: Barras apiladas al 100% por región y tipo ──

chart_proporcional = alt.Chart(agrupado).mark_bar().encode(
    x=alt.X('region:N',
            sort=orden_regiones,
            title='Región',
            axis=alt.Axis(labelAngle=-45, labelLimit=200)),
    y=alt.Y('cantidad:Q',
            stack='normalize',  # <--- ESTO hace que todas las barras sumen 100%
            title='Proporción de tipos de glaciares',
            axis=alt.Axis(format='%')), # Formato de porcentaje en el eje Y
    color=alt.Color('tipo_glaciar:N',
                    title='Tipo de glaciar',
                    scale=alt.Scale(
                        domain=list(colores.keys()),
                        range=list(colores.values())
                    )),
    tooltip=[
        alt.Tooltip('region:N', title='Región'),
        alt.Tooltip('tipo_glaciar:N', title='Tipo'),
        alt.Tooltip('cantidad:Q', title='Cantidad absoluta'),
        alt.Tooltip('area_total_km2:Q', title='Área total (km²)', format='.2f')
    ]
).properties(
    title=alt.TitleParams(
        text='Composición porcentual de glaciares por región en Chile (2022)',
        subtitle='Cada barra representa el 100% de los glaciares de la región seleccionada',
        fontSize=16,
        anchor='start'
    ),
    width=750,
    height=420
)

chart_proporcional